# Prepare Extra Artifacts (Tuần 9-10)

Notebook này **CHỈ chạy 1 lần**, trước khi mở dashboard Streamlit.

Trong `Analysis_model.ipynb`, phần "PHÂN VÙNG KHÁCH HÀNG" (KMeans + PCA)
đã được train nhưng **chưa lưu ra file**. Notebook này train lại đúng các
bước đó (RobustScaler → MiniBatchKMeans k=5 → PCA 2 chiều) và lưu kết quả
ra ổ đĩa, để `app.py` (Streamlit) load lên dùng.

**Lưu ý quan trọng:** hãy mở notebook này với **1 kernel MỚI, sạch**
(không chạy chung kernel với `Analysis_model.ipynb`), để tránh bị đầy RAM
và crash kernel do các biến/model nặng (LSTM, dataframe cũ...) còn tồn
trong bộ nhớ. Nếu VS Code tự chọn kernel đang chạy sẵn, vào góc trên bên
phải notebook → chọn kernel → **Restart** hoặc chọn 1 kernel Python khác.

**Đặt file này trong thư mục `Notebooks/`** (cùng chỗ với
`FraudDetectionPipeline_RF.pkl`, `threshold.json`), vì nó dùng chung
đường dẫn dữ liệu `../Data/AIML Dataset.csv` giống notebook gốc.

Chạy các cell theo thứ tự lần lượt (Shift + Enter). Sau khi chạy xong sẽ
có thêm 4 file: `cluster_scaler.pkl`, `kmeans_model.pkl`, `pca_model.pkl`,
`pca_background_sample.csv`.

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

print("Đã import xong thư viện.")

## 2. Cấu hình đường dẫn

In [ ]:
# ĐƯỜNG DẪN DỮ LIỆU - giống hệt Analysis_model.ipynb
DATA_PATH = "../Data/AIML Dataset.csv"

CLUSTER_FEATURES = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

# CHỈ đọc đúng các cột cần dùng - bỏ nameOrig/nameDest (2 cột chuỗi text
# chiếm rất nhiều RAM vì có ~6 triệu giá trị khác nhau) để tránh crash
# kernel do hết bộ nhớ (OOM).
USE_COLS = CLUSTER_FEATURES + ["isFraud"]

print("Các cột sẽ đọc:", USE_COLS)

## 3. Đọc dữ liệu gốc (chỉ các cột cần thiết)

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=USE_COLS)

# Ép kiểu float32 thay vì float64 mặc định để giảm ~50% bộ nhớ sử dụng
for col in CLUSTER_FEATURES:
    df[col] = df[col].astype("float32")

X_cluster = df[CLUSTER_FEATURES]

print("Đã đọc xong dữ liệu. Số dòng:", len(df))
df.head()

## 4. Chuẩn hoá dữ liệu bằng RobustScaler

In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("Đã chuẩn hoá xong. Shape:", X_scaled.shape)

## 5. Huấn luyện KMeans (k=5) - giống notebook gốc

In [ ]:
kmeans = MiniBatchKMeans(
    n_clusters=5,
    batch_size=2048,
    random_state=42,
    n_init=10,
)
cluster_labels = kmeans.fit_predict(X_scaled)

print("Đã phân cụm xong. Số lượng mỗi cụm:")
print(pd.Series(cluster_labels).value_counts().sort_index())

## 6. Huấn luyện PCA (giảm về 2 chiều để vẽ biểu đồ)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Đã giảm chiều xong. Shape:", X_pca.shape)

## 7. Lưu model ra file (.pkl)

In [ ]:
joblib.dump(scaler, "cluster_scaler.pkl")
joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(pca, "pca_model.pkl")

print("Đã lưu: cluster_scaler.pkl, kmeans_model.pkl, pca_model.pkl")

## 8. Tạo mẫu dữ liệu nền để vẽ biểu đồ Plotly (không cần load 6 triệu dòng mỗi lần mở dashboard)

In [ ]:
bg = pd.DataFrame(X_pca, columns=["PCA1", "PCA2"])
bg["Cluster"] = cluster_labels
bg["isFraud"] = df["isFraud"].values

bg_sample = bg.sample(n=min(8000, len(bg)), random_state=42)
bg_sample.to_csv("pca_background_sample.csv", index=False)

print(f"Đã lưu: pca_background_sample.csv ({len(bg_sample)} dòng mẫu)")

## 9. Hoàn tất ✅

Bây giờ có thể mở terminal ở thư mục gốc dự án và chạy:

```bash
streamlit run app.py
```

In [ ]:
print("HOÀN TẤT! Kiểm tra 4 file .pkl/.csv vừa tạo trong thư mục này.")
print("Bây giờ có thể chạy: streamlit run app.py (từ thư mục gốc dự án)")